In [1]:
from Simulation.system_functions import PolymerCSTR
import numpy as np
import os
from utils.td3_helpers import load_and_prepare_system_data
from TD3Agent.reward_functions import make_reward_fn_mpc_quadratic, make_reward_fn_relative_QR
from utils.scaling_helpers import apply_min_max
from Simulation.mpc_run import run_mpc
from Plotting_fns.mpc_plot_fns import plot_mpc_results_cstr
import pickle

## Initialize the system

In [2]:
# First initiate the system
# Parameters
Ad = 2.142e17           # h^-1
Ed = 14897              # K
Ap = 3.816e10           # L/(molh)
Ep = 3557               # K
At = 4.50e12            # L/(molh)
Et = 843                # K
fi = 0.6                # Coefficient
m_delta_H_r = -6.99e4   # j/mol
hA = 1.05e6             # j/(Kh)
rhocp = 1506            # j/(Kh)
rhoccpc = 4043          # j/(Kh)
Mm = 104.14             # g/mol
system_params = np.array([Ad, Ed, Ap, Ep, At, Et, fi, m_delta_H_r, hA, rhocp, rhoccpc, Mm])

In [3]:
# Design Parameters
CIf = 0.5888    # mol/L
CMf = 8.6981    # mol/L
Qi = 108.       # L/h
Qs = 459.       # L/h
Tf = 330.       # K
Tcf = 295.      # K
V = 3000.       # L
Vc = 3312.4     # L

system_design_params = np.array([CIf, CMf, Qi, Qs, Tf, Tcf, V, Vc])

In [4]:
# Steady State Inputs
Qm_ss = 378.    # L/h
Qc_ss = 471.6   # L/h

system_steady_state_inputs = np.array([Qc_ss, Qm_ss])

In [5]:
# Sampling time of the system
delta_t = 0.5 # 30 mins

In [6]:
# Initiate the CSTR for steady state values
cstr = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)
steady_states={"ss_inputs":cstr.ss_inputs,
               "y_ss":cstr.y_ss}

## Loading the system matrices, min max scaling, and min max of the states

In [7]:
import numpy as np
import control
from scipy import signal


class MpcSolver(object):
    def __init__(self, A, B, C, Q_out, R_in, NP, NC, D=None):
        self.A = np.asarray(A, float)
        self.B = np.asarray(B, float)
        self.C = np.asarray(C, float)
        self.D = None if D is None else np.asarray(D, float)

        self.NP = int(NP)
        self.NC = int(NC)

        self.Q_out = np.asarray(Q_out, float).reshape(-1)
        self.R_in = np.asarray(R_in, float).reshape(-1)

    def mpc_opt_fun(self, x, y_sp, u_prev_dev, x0_model):
        n_inputs = self.B.shape[1]
        n_outputs = self.C.shape[0]

        U = np.asarray(x[:n_inputs * self.NC], float).reshape(self.NC, n_inputs)

        y_sp = np.asarray(y_sp, float).reshape(n_outputs,)
        u_prev_dev = np.asarray(u_prev_dev, float).reshape(n_inputs,)
        x0_model = np.asarray(x0_model, float)

        x_pred = np.zeros((self.A.shape[0], self.NP + 1), dtype=float)
        x_pred[:, 0] = x0_model

        for j in range(self.NP):
            idx = j if j < self.NC else self.NC - 1
            x_pred[:, j + 1] = self.A @ x_pred[:, j] + self.B @ U[idx, :]

        y_pred = self.C @ x_pred
        y_dev = y_pred[:, 1:] - y_sp[:, None]

        U_prev = np.vstack([u_prev_dev.reshape(1, -1), U[:-1, :]])
        du = U - U_prev

        obj = 0.0
        for i in range(n_outputs):
            obj += float(self.Q_out[i]) * float(np.sum(y_dev[i, :] ** 2))
        for j in range(n_inputs):
            obj += float(self.R_in[j]) * float(np.sum(du[:, j] ** 2))

        return float(obj)


def augment_state_space_rawlings(A, B, C, Bd=None, Cd=None):
    """
    Rawlings-style offset-free augmentation:

        x_{k+1} = A x_k + B u_k + Bd d_k
        d_{k+1} = d_k
        y_k = C x_k + Cd d_k

    Parameters
    ----------
    A : np.ndarray, shape (n, n)
    B : np.ndarray, shape (n, m)
    C : np.ndarray, shape (p, n)
    Bd : np.ndarray or None, shape (n, nd)
        Disturbance-to-state matrix.
        If None, use output-disturbance default with nd = p and Bd = 0.
    Cd : np.ndarray or None, shape (p, nd)
        Disturbance-to-output matrix.
        If None, use output-disturbance default with nd = p and Cd = I.

    Returns
    -------
    A_aug : np.ndarray, shape (n + nd, n + nd)
    B_aug : np.ndarray, shape (n + nd, m)
    C_aug : np.ndarray, shape (p, n + nd)
    Bd : np.ndarray
    Cd : np.ndarray
    """
    A = np.asarray(A, float)
    B = np.asarray(B, float)
    C = np.asarray(C, float)

    n = A.shape[0]
    m = B.shape[1]
    p = C.shape[0]

    if A.shape != (n, n):
        raise ValueError("A must be square.")
    if B.shape[0] != n:
        raise ValueError("B row dimension must match A.")
    if C.shape[1] != n:
        raise ValueError("C column dimension must match A.")

    if Bd is None and Cd is None:
        nd = p
        Bd = np.zeros((n, nd), dtype=float)
        Cd = np.eye(p, dtype=float)
    elif Bd is None or Cd is None:
        raise ValueError("Provide both Bd and Cd, or neither.")
    else:
        Bd = np.asarray(Bd, float)
        Cd = np.asarray(Cd, float)
        nd = Bd.shape[1]

        if Bd.shape[0] != n:
            raise ValueError("Bd row dimension must match A.")
        if Cd.shape[0] != p:
            raise ValueError("Cd row dimension must match C.")
        if Cd.shape[1] != nd:
            raise ValueError("Cd column dimension must match Bd.")

    A_aug = np.block([
        [A, Bd],
        [np.zeros((nd, n), dtype=float), np.eye(nd, dtype=float)],
    ])
    B_aug = np.vstack([B, np.zeros((nd, m), dtype=float)])
    C_aug = np.hstack([C, Cd])

    return A_aug, B_aug, C_aug, Bd, Cd


def hautus_detectable_discrete(A, C, tol=1e-9):
    """
    Discrete-time detectability test using Hautus:
    check only eigenvalues with |lambda| >= 1 - tol.
    """
    A = np.asarray(A, float)
    C = np.asarray(C, float)

    n = A.shape[0]
    eigvals = np.linalg.eigvals(A)

    bad = []
    for lam in eigvals:
        if abs(lam) >= 1.0 - tol:
            M = np.vstack([lam * np.eye(n) - A, C])
            r = np.linalg.matrix_rank(M, tol)
            if r < n:
                bad.append((lam, r))

    return len(bad) == 0, bad


def rawlings_rank_matrix(A, C, Bd, Cd):
    """
    Build the Rawlings rank-check matrix:
        [I - A   Bd]
        [ C     Cd]
    """
    A = np.asarray(A, float)
    C = np.asarray(C, float)
    Bd = np.asarray(Bd, float)
    Cd = np.asarray(Cd, float)

    n = A.shape[0]
    M = np.block([
        [np.eye(n) - A, Bd],
        [C, Cd],
    ])
    return M


def check_rawlings_augmentation(A, B, C, Bd, Cd, tol=1e-9, verbose=True):
    """
    Check the two key conditions:
    1) unaugmented (A, C) detectable
    2) rank([I - A, Bd; C, Cd]) = n + nd
    """
    A = np.asarray(A, float)
    B = np.asarray(B, float)
    C = np.asarray(C, float)
    Bd = np.asarray(Bd, float)
    Cd = np.asarray(Cd, float)

    n = A.shape[0]
    nd = Bd.shape[1]

    det_ok, det_bad = hautus_detectable_discrete(A, C, tol=tol)

    M = rawlings_rank_matrix(A, C, Bd, Cd)
    rank_M = np.linalg.matrix_rank(M, tol)
    rank_ok = rank_M == (n + nd)

    if verbose:
        print("Unaugmented detectability OK:", det_ok)
        if not det_ok:
            print("Failed Hautus modes (lambda, rank):", det_bad)

        print("Rawlings rank matrix shape:", M.shape)
        print("Rawlings rank =", rank_M, "| target =", n + nd)
        print("Rawlings rank condition OK:", rank_ok)

    return {
        "detectable_unaugmented": det_ok,
        "detectability_fail_modes": det_bad,
        "rawlings_rank": rank_M,
        "rawlings_rank_target": n + nd,
        "rawlings_rank_ok": rank_ok,
        "M_rank": M,
    }


def compute_observer_gain(A, C, desired_poles):
    obs_gain_calc = signal.place_poles(A.T, C.T, desired_poles, method="KNV0")
    L = np.squeeze(obs_gain_calc.gain_matrix).T

    observability_matrix = control.obsv(A, C)
    rank = np.linalg.matrix_rank(observability_matrix)
    if rank == A.shape[0]:
        print("The system is observable.")
    else:
        print("The system is not observable. Rank:", rank, "of", A.shape[0])

    return L


def try_rawlings_candidates(A, B, C, candidates, tol=1e-9):
    """
    candidates: dict of {"name": (Bd, Cd)}
    """
    results = {}
    for name, (Bd, Cd) in candidates.items():
        print("\n" + "=" * 70)
        print("Candidate:", name)
        res = check_rawlings_augmentation(A, B, C, Bd, Cd, tol=tol, verbose=True)
        results[name] = res
    return results

In [8]:
dir_path = os.path.join(os.getcwd(), "Data")

In [9]:
system_dict_path = os.path.join(dir_path, "system_dict")
with open(system_dict_path, "rb") as file:
    system_dict = pickle.load(file)

A = system_dict["A"]
B = system_dict["B"]
C = system_dict["C"]

In [10]:
p = C.shape[0]
n = A.shape[0]

candidates = {
    "output_disturbance": (
        np.zeros((n, p)),
        np.eye(p),
    ),
    "state_disturbance_via_B": (
        B.copy(),
        np.zeros((p, p)),
    ),
    "mixed_B_I": (
        B.copy(),
        np.eye(p),
    ),
    "mixed_0.1B_I": (
        0.1 * B.copy(),
        np.eye(p),
    ),
    "mixed_0.5B_I": (
        0.5 * B.copy(),
        np.eye(p),
    ),
}

cand_results = try_rawlings_candidates(A, B, C, candidates, tol=1e-9)


Candidate: output_disturbance
Unaugmented detectability OK: True
Rawlings rank matrix shape: (9, 9)
Rawlings rank = 9 | target = 9
Rawlings rank condition OK: True

Candidate: state_disturbance_via_B
Unaugmented detectability OK: True
Rawlings rank matrix shape: (9, 9)
Rawlings rank = 9 | target = 9
Rawlings rank condition OK: True

Candidate: mixed_B_I
Unaugmented detectability OK: True
Rawlings rank matrix shape: (9, 9)
Rawlings rank = 9 | target = 9
Rawlings rank condition OK: True

Candidate: mixed_0.1B_I
Unaugmented detectability OK: True
Rawlings rank matrix shape: (9, 9)
Rawlings rank = 9 | target = 9
Rawlings rank condition OK: True

Candidate: mixed_0.5B_I
Unaugmented detectability OK: True
Rawlings rank matrix shape: (9, 9)
Rawlings rank = 9 | target = 9
Rawlings rank condition OK: True


In [11]:
p = C.shape[0]

Bd = B.copy()
Cd = np.eye(p)

A_aug, B_aug, C_aug, Bd_used, Cd_used = augment_state_space_rawlings(
    A, B, C, Bd=Bd, Cd=Cd
)

In [12]:
# Defining the range of setpoints for data generation
setpoint_y = np.array([[2.8, 320.],
                       [5., 326.]])
u_min = np.array([71.6, 78])
u_max = np.array([870, 670])

system_data = load_and_prepare_system_data(steady_states=steady_states, setpoint_y=setpoint_y, u_min=u_min, u_max=u_max)

In [13]:
# A_aug = system_data["A_aug"]
# B_aug = system_data["B_aug"]
# C_aug = system_data["C_aug"]

In [14]:
data_min = system_data["data_min"]
data_max = system_data["data_max"]

In [15]:
min_max_states = {'max_s': np.array([256.79686253, 256.01560603,  48.99447186, 144.79949103,
          2.82199733,   3.14014989,   2.78866348,   3.71691422,
          6.2029936 ]),
                  'min_s': np.array([ -272.28060121, -1112.33972595,   -76.63993491,  -608.60327886,
           -3.94399122,    -3.93115257,    -2.9532091 ,    -4.06547624,
          -28.25906582])}

In [16]:
y_sp_scaled_deviation = system_data["y_sp_scaled_deviation"]

In [17]:
b_min = system_data["b_min"]
b_max = system_data["b_max"]

In [18]:
min_max_dict = system_data["min_max_dict"]
min_max_dict["x_max"] = np.array([256.79686253, 256.01560603,  48.99447186, 144.79949103,
          2.82199733,   3.14014989,   2.78866348,   3.71691422,
          6.2029936 ])
min_max_dict["x_min"] = np.array([ -272.28060121, -1112.33972595,   -76.63993491,  -608.60327886,
           -3.94399122,    -3.93115257,    -2.9532091 ,    -4.06547624,
          -28.25906582])

In [19]:
# Setpoints in deviation form
inputs_number = int(B_aug.shape[1])
y_sp_scenario = np.array([[4.5, 324],
                          [3.4, 321]])

y_sp_scenario = (apply_min_max(y_sp_scenario, data_min[inputs_number:], data_max[inputs_number:])
                 - apply_min_max(steady_states["y_ss"], data_min[inputs_number:], data_max[inputs_number:]))
n_tests = 10
set_points_len = 400
TEST_CYCLE = [False, False]
warm_start = 0
ACTOR_FREEZE = 0
warm_start_plot = warm_start * 2 * set_points_len + ACTOR_FREEZE

In [20]:
# Observer Gain
poles = np.array(np.array([0.44619852, 0.33547649, 0.36380595, 0.70467118, 0.3562966,
                           0.42900673, 0.4228262 , 0.96916776, 0.91230187]))
check_res = check_rawlings_augmentation(
    A, B, C, Bd_used, Cd_used, tol=1e-9, verbose=True
)

det_aug_ok, det_aug_bad = hautus_detectable_discrete(A_aug, C_aug, tol=1e-9)
print("Augmented detectability OK:", det_aug_ok)
print("Bad augmented modes:", det_aug_bad)
L = compute_observer_gain(A_aug, C_aug, poles)

Unaugmented detectability OK: True
Rawlings rank matrix shape: (9, 9)
Rawlings rank = 9 | target = 9
Rawlings rank condition OK: True
Augmented detectability OK: True
Bad augmented modes: []
The system is observable.


C:\Users\HAMEDI\AppData\Local\Temp\ipykernel_40116\1278753071.py:204: UserWarning: Convergence was not reached after maxiter iterations.
You asked for a tolerance of 0.001, we got 0.9999999356032435.
  obs_gain_calc = signal.place_poles(A.T, C.T, desired_poles, method="KNV0")


# MPC Config.

In [127]:
import importlib
import standard_lyap_tracking_mpc
importlib.reload(standard_lyap_tracking_mpc)

from standard_lyap_tracking_mpc import (
    design_standard_tracking_terminal_ingredients,
    StandardTrackingLyapunovMpcRawlingsTargetSolver,
    run_standard_tracking_lyapunov_mpc_rawlings_target,
)
# import importlib
# import standard_lyap_tracking_mpc_v2
# importlib.reload(standard_lyap_tracking_mpc_v2)
#
# from standard_lyap_tracking_mpc_v2 import (
#     design_standard_tracking_terminal_ingredients,
#     StandardTrackingLyapunovMpcRawlingsTargetSolver,
#     run_standard_tracking_lyapunov_mpc_rawlings_target,
# )

In [128]:
# MPC parameters
n_inputs = 2
predict_h = 9
cont_h = 3
u_ss = apply_min_max(steady_states['ss_inputs'], data_min[:n_inputs], data_max[:n_inputs])
b_min = apply_min_max(np.array([71.6, 78]), data_min[:n_inputs], data_max[:n_inputs])
b_max= apply_min_max(np.array([870, 670]), data_min[:n_inputs], data_max[:n_inputs])
b1 = (b_min[0]-u_ss[0], b_max[0]-u_ss[0])
b2 = (b_min[1]-u_ss[1], b_max[1]-u_ss[1])
bnds = (b1, b2)*cont_h
cons = []
IC_opt = np.zeros(n_inputs*cont_h)
Q1_penalty = 5.
Q2_penalty = 1.
R1_penalty = 1
R2_penalty = 1

In [129]:
Qy_diag_std = np.array([Q1_penalty, Q2_penalty], dtype=float)

# This is the penalty on u - u_s in the standard tracking MPC.
# Start small but nonzero.
Su_diag_std = np.array([0.1, 0.1], dtype=float)

P_x, K_x, term_dbg = design_standard_tracking_terminal_ingredients(
    A_aug=A_aug,
    B_aug=B_aug,
    C_aug=C_aug,
    Qy_diag=Qy_diag_std,
    Su_diag=Su_diag_std,
    u_min=np.array([bnds[0][0], bnds[1][0]], dtype=float),
    u_max=np.array([bnds[0][1], bnds[1][1]], dtype=float),
    return_debug=True,
)

print("Closed-loop terminal eigenvalues:")
print(term_dbg["eig_cl"])
print("K_x:")
print(K_x)

Closed-loop terminal eigenvalues:
[ 0.00000000e+00+0.00000000e+00j  8.85807104e-01+0.00000000e+00j
  9.36687321e-01+0.00000000e+00j  9.27699034e-01+0.00000000e+00j
  9.46390924e-01+0.00000000e+00j -6.27598885e-12+3.57626464e-10j
 -6.27598885e-12-3.57626464e-10j]
K_x:
[[-3.86292014e-04 -2.90646814e-03 -4.76650600e-04  2.50174533e-03
  -0.00000000e+00 -1.59185483e-06 -2.86796382e-06]
 [-1.08899476e-03  5.02585993e-03 -2.63784407e-03 -4.34617978e-03
  -0.00000000e+00 -2.17235569e-02 -1.92400409e-02]]


In [130]:
NP_std = predict_h
NC_std = cont_h

Qy_diag_std = np.array([Q1_penalty, Q2_penalty], dtype=float)
Su_diag_std = np.array([0., 0.], dtype=float)
Rdu_diag_std = np.array([R1_penalty, R2_penalty], dtype=float)

LMPC_std = StandardTrackingLyapunovMpcRawlingsTargetSolver(
    A_aug=A_aug,
    B_aug=B_aug,
    C_aug=C_aug,
    Qy_diag=Qy_diag_std,
    Su_diag=Su_diag_std,
    NP=NP_std,
    NC=NC_std,
    P_x=P_x,
    K_x=K_x,
    Rdu_diag=Rdu_diag_std,
    terminal_set_on=False,
    terminal_alpha_scale=0.0,
    terminal_cost_scale=0.005,
)

## Reward configuration

In [131]:
n_inputs = 2

dy = data_max[n_inputs:] - data_min[n_inputs:]
y_sp_nom = 0.5 * (data_min[n_inputs:] + data_max[n_inputs:])

k_rel = np.array([0.003, 0.0003])
band_floor_phys = np.array([0.006, 0.07])

band_phys = np.maximum(k_rel * np.abs(y_sp_nom), band_floor_phys)

scale_factor = 1.0  # use 2.0 for [-1, 1] scaling, 1.0 for [0, 1]
band_scaled = scale_factor * band_phys / dy

q0 = 1.4
Q_diag = q0 / np.maximum(band_scaled ** 2, 1e-12)

print("dy:", dy)
print("y_sp_nom:", y_sp_nom)
print("band_phys:", band_phys)
print("band_scaled:", band_scaled)
print("Q_diag:", Q_diag)

dy: [0.22165278 0.78153727]
y_sp_nom: [  3.83915067 323.21371982]
band_phys: [0.01151745 0.09696412]
band_scaled: [0.05196169 0.12406845]
Q_diag: [518.51529284  90.95055189]


In [132]:
Q_diag = np.array([518., 90.])          # rounded from the band-based calculation
R_diag = np.array([90., 90.])          # move cost for du_scaled ~ 0.02

n_inputs = 2

print("Band scaled are:")

params, reward_fn = make_reward_fn_relative_QR(
    data_min, data_max, n_inputs,
    k_rel, band_floor_phys,
    Q_diag, R_diag,
    tau_frac=0.7,
    gamma_out=0.5, gamma_in=0.5,
    beta=7.0, gate="geom", lam_in=1.0,
    bonus_kind="exp", bonus_k=12.0, bonus_p=0.6, bonus_c=20.0,
)
print(params)

Band scaled are:
{'k_rel': array([0.003 , 0.0003]), 'band_floor_phys': array([0.006, 0.07 ]), 'band_floor_scaled': array([0.02706937, 0.08956707]), 'Q_diag': array([518.,  90.]), 'R_diag': array([90., 90.]), 'tau_frac': 0.7, 'gamma_out': 0.5, 'gamma_in': 0.5, 'beta': 7.0, 'gate': 'geom', 'lam_in': 1.0, 'bonus_kind': 'exp', 'bonus_k': 12.0, 'bonus_p': 0.6, 'bonus_c': 20.0}


In [133]:
params_qp, reward_fn_qp = make_reward_fn_mpc_quadratic(Q_diag=np.array([5., 1.]), R_diag=np.array([1., 1.]))
print(params_qp)

{'Q_diag': array([5., 1.]), 'R_diag': array([1., 1.])}


# Disturbance Profile

In [134]:
nominal_qs = 459
nominal_qi = 108
nominal_hA = 1.05e6
qi_change = 0.95
qs_change = 1.05
ha_change = 0.92

# Environment Run nominal

In [123]:
cstr = PolymerCSTR(
    system_params,
    system_design_params,
    system_steady_state_inputs,
    delta_t
)

results_std = run_standard_tracking_lyapunov_mpc_rawlings_target(
    system=cstr,
    LMPC_obj=LMPC_std,
    y_sp_scenario=y_sp_scenario,
    n_tests=n_tests,
    set_points_len=set_points_len,
    steady_states=steady_states,
    IC_opt=IC_opt,
    bnds=bnds,
    L=L,
    data_min=data_min,
    data_max=data_max,
    test_cycle=TEST_CYCLE,
    reward_fn=reward_fn_qp,
    nominal_qi=nominal_qi,
    nominal_qs=nominal_qs,
    nominal_ha=nominal_hA,
    qi_change=qi_change,
    qs_change=qs_change,
    ha_change=ha_change,
    Qs_tgt_diag=np.array([100.0, 100.0], dtype=float),
    Ru_tgt_diag=np.array([1.0, 1.0], dtype=float),
    u_nom_tgt=np.zeros(B_aug.shape[1], dtype=float),
    w_x_tgt=1e-6,
    mode="nominal",
    skip_terminal_if_alpha_small = True,
    disturbance_after_step = True,
    use_external_target_for_tracking = False,
    use_target_on_solver_fail = False,
    # Ty_tgt_diag = np.array([100.0, 100.0], dtype=float),
    # Qx_tgt_diag = None,
    # Qdx_tgt_diag = np.array([0.01] * (A_aug.shape[0] - C_aug.shape[0]), dtype=float),
    # Rdu_tgt_diag = np.array([0.05, 0.05], dtype=float),
    # u_tight_tgt = np.zeros(B_aug.shape[1], dtype=float),
    # y_tight_tgt = np.zeros(C_aug.shape[0], dtype=float)
)

Sub_Episode: 1 | avg. reward: -24.221924009528855 | method: standard_tracking_lyapunov_mpc_rawlings_target | success: True | alpha_terminal: 1.0 | terminal_margin: 0.9998439983595673 | target_slack_inf: 1.2036139014281924 | nit: 4
Sub_Episode: 2 | avg. reward: -27.669531517906016 | method: standard_tracking_lyapunov_mpc_rawlings_target | success: True | alpha_terminal: 1.0 | terminal_margin: 0.9998434287088704 | target_slack_inf: 1.2074631840749273 | nit: 4
Sub_Episode: 3 | avg. reward: -27.647808080418418 | method: standard_tracking_lyapunov_mpc_rawlings_target | success: True | alpha_terminal: 1.0 | terminal_margin: 0.9998426465005863 | target_slack_inf: 1.2109981454493126 | nit: 2
Sub_Episode: 4 | avg. reward: -27.627694658447513 | method: standard_tracking_lyapunov_mpc_rawlings_target | success: True | alpha_terminal: 1.0 | terminal_margin: 0.9998424606384685 | target_slack_inf: 1.214185398803699 | nit: 2
Sub_Episode: 5 | avg. reward: -27.6099681396639 | method: standard_tracking_l

In [124]:
(
    y_std,
    u_std,
    avg_rewards_std,
    rewards_std,
    xhatdhat_std,
    nFE_std,
    time_in_sub_episodes_std,
    y_sp_std,
    yhat_std,
    delta_y_storage_std,
    delta_u_storage_std,
    lmpc_info_storage_std,
    target_info_storage_std,
) = results_std

# Saving the results and plotting

In [126]:
import sys
if "/mnt/data" not in sys.path:
    sys.path.append("/mnt/data")

from target_selector_diagnostics import export_target_selector_diagnostics

out_dir = export_target_selector_diagnostics(
    target_info_storage=target_info_storage_std,
    lmpc_info_storage=lmpc_info_storage_std,
    out_dir="Result/target_diag_stage2",
    prefix_name="target_stage2",
    delta_t=delta_t,
)

print(out_dir)

C:\Users\HAMEDI\OneDrive - McMaster University\PythonProjects\Polymer_example\target_selector_diagnostics.py:164: RuntimeWarning: Mean of empty slice
  'target_error_inf_mean': float(np.nanmean(traces['target_error_inf'])),
C:\Users\HAMEDI\OneDrive - McMaster University\PythonProjects\Polymer_example\target_selector_diagnostics.py:165: RuntimeWarning: All-NaN slice encountered
  'target_error_inf_max': float(np.nanmax(traces['target_error_inf'])),


Result/target_diag_stage2\target_stage2\20260323_213253


In [121]:
# from run_standard_lyap_export import save_standard_lyap_results
#
# export_info = save_standard_lyap_results(
#     results_std,
#     out_dir="Result/standard_lyap_case1",
#     delta_t=delta_t,
#     output_labels=("eta", "T"),
#     input_labels=("Qc", "Qm"),
#     extra_arrays={
#         "A_aug": A_aug,
#         "B_aug": B_aug,
#         "C_aug": C_aug,
#         "L": L,
#     },
#     meta={
#         "case_name": "standard_lyap_case1",
#     },
#     last_window=time_in_sub_episodes_std,
# )
#
# print(export_info)

In [125]:
out_dir = plot_mpc_results_cstr(
    y_sp_std, steady_states, nFE_std, delta_t, time_in_sub_episodes_std,
    y_std, u_std, data_min, data_max,
    directory=dir_path, prefix_name="lmpc_result_nominal"
)
print("Saved to:", out_dir)

Saved to: C:\Users\HAMEDI\OneDrive - McMaster University\PythonProjects\Polymer_example\Data\lmpc_result_nominal\20260323_211705
